Project : Pentaho Log Intelligence

Layer   : Bronze

Notebook: 01_Bronze_Ingestion

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Bronze Delta table.

Author: Ernesto Felipe Garay Cervantes






#### RECIBIMIENTO DE PARAMETROS

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo)

 
#dbutils.notebook.exit("notebook hijo parametro que recibe : " + archivo)

### Configuración 

In [0]:
#configuration
CATALOG = "pentaho_logs"
SCHEMA = "bronze"

VOLUME_PATH = "/Volumes/pentaho_logs/bronze/volume_cartelogs"
BRONZE_TABLE_CARTE = f"{CATALOG}.{SCHEMA}.bronze_logs_carte"



### Validación de Volumen

In [0]:
display(dbutils.fs.ls(VOLUME_PATH))


### Lectura de archivos de carte

In [0]:
from pyspark.sql.functions import col

VOLUME_PATH = "/Volumes/pentaho_logs/bronze/volume_cartelogs"

archivos_path = [
    f"{VOLUME_PATH}/{archivo}"
    for archivo in archivos_nuevos
]

print("==== PATH PROXIMOS A PROCESARSE ====")

for path in archivos_path:
    print(path)

df_bronze = (
    spark.read
    .text(archivos_path)
    .select(
        col("value").alias("raw_text"),
        col("_metadata.file_path").alias("file_path"),
        col("_metadata.file_name").alias("file_name")
    )
)

#display(df_bronze.limit(10))




### Enriquecimiento DataFrame Carte

In [0]:
from pyspark.sql.functions import  regexp_extract, to_date, current_timestamp,col
df_bronze = (df_bronze.withColumn("application",regexp_extract("file_name", r"^([A-Za-z]+)_", 1))
                      .withColumn("server_port",regexp_extract("file_name", r"_(\d+)-", 1).cast("int"))
                      .withColumn("log_date",to_date(regexp_extract("file_name", r"(\d{4}-\d{2}-\d{2})", 1)))
                      .withColumn("file_sequence",regexp_extract("file_name", r"-(\d+)\.log$", 1).cast("int"))
                      .withColumn("Timestamp",current_timestamp())
             )

#display(df_bronze.limit(20))



### Creación tabla BRONZE_TABLE_CARTE

In [0]:
BRONZE_TABLE_CARTE = "pentaho_logs.bronze.bronze_logs_carte"
(
   df_bronze.write
        .format("delta")
       .mode("append")  #para sobreescribir con la tabla y no crear
       .saveAsTable(BRONZE_TABLE_CARTE)
)

In [0]:
#display(spark.table(BRONZE_TABLE_CARTE).limit(20))
